In [ ]:
from sqlalchemy import create_engine
from snowflake.sqlalchemy import URL
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.decomposition import PCA
from scipy.stats import ks_2samp, linregress, median_abs_deviation
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [ ]:
engine = create_engine(URL(
    account       = "UHG-UHGDWAAS",
    user          = "KHANG.NGUYEN@UHC.COM",
    authenticator = "externalbrowser",
    role          = "AZU_SDRP_VING_PRD_DEVELOPER_ROLE",
    warehouse     = "VING_PRD_MNR_HCE_DATAINFRA_WH",
    database      = "VING_PRD_TREND_DB",
    schema        = "TMP_1M"
))

In [ ]:
# hospital_group x market aggregation — MNR FFS, 2025
df = pd.read_sql("""
    select
        d.collection as hospital_group
        , a.fin_market
        , count(*) as case_count_2025
        , sum(a.initialfulladr_cases) as initial_adr_cnt_2025
        , sum(a.persistentfulladr_cases) as persistent_adr_cnt
        , sum(a.p2p_full_ovtn) as p2p_ovrtn_cnt
        , sum(a.appeal_ovrtn_ind) as appeal_ovrtn_cnt
        , sum(a.mcr_ovtrn_ind) as mcr_ovrtn_cnt
        , sum(a.icm_md_reviewed_ind) as md_reviewed_cnt
    from tmp_1m.ec_avtar_25_26_3_od as a
    left join tmp_1y.tin_collection as d
        on substr(a.fa_prov_id, 2, 9) = d.tin
    where fin_brand in ('M&R', 'C&S')
        and (
            (ip_type in ('Medical', 'Surgical', 'Transplant')
             and to_varchar(admit_dt_act, 'MM/dd/yyyy') is not null)
            or ip_type in ('LTAC', 'SNF', 'AIR')
        )
        and fin_brand = 'M&R'
        and loc_flag = 1
        and (
            (global_cap = 'NA' and sgr_source_name = 'COSMOS'
             and fin_product_level_3 != 'INSTITUTIONAL' and tfm_include_flag = 1)
            or (sgr_source_name = 'NICE' and nce_tadm_dec_risk_type in ('FFS', 'PHYSICIAN'))
        )
        and hce_admit_month between '202501' and '202512'
    group by 1, 2
    having count(*) >= 30
""", engine)
print(f"Core (2025): {df.shape}")

In [ ]:
# member_appeal only exists from 202601 onward — separate pull
df_ma = pd.read_sql("""
    select
        d.collection as hospital_group
        , a.fin_market
        , count(*) as case_count_2026
        , sum(a.initialfulladr_cases) as initial_adr_cnt_2026
        , sum(a.member_appeal_ind) as member_appeal_cnt
        , sum(a.member_appeal_ovtn_ind) as member_appeal_ovtn_cnt
    from tmp_1m.ec_avtar_25_26_3_od as a
    left join tmp_1y.tin_collection as d
        on substr(a.fa_prov_id, 2, 9) = d.tin
    where fin_brand in ('M&R', 'C&S')
        and (
            (ip_type in ('Medical', 'Surgical', 'Transplant')
             and to_varchar(admit_dt_act, 'MM/dd/yyyy') is not null)
            or ip_type in ('LTAC', 'SNF', 'AIR')
        )
        and fin_brand = 'M&R'
        and loc_flag = 1
        and (
            (global_cap = 'NA' and sgr_source_name = 'COSMOS'
             and fin_product_level_3 != 'INSTITUTIONAL' and tfm_include_flag = 1)
            or (sgr_source_name = 'NICE' and nce_tadm_dec_risk_type in ('FFS', 'PHYSICIAN'))
        )
        and hce_admit_month >= '202601'
    group by 1, 2
    having sum(a.initialfulladr_cases) >= 10
""", engine)
print(f"Member appeal (202601+): {df_ma.shape}")

In [ ]:
# monthly grain for trend detection (Section F) — 202301 onward
df_monthly = pd.read_sql("""
    select
        d.collection as hospital_group
        , a.fin_market
        , a.hce_admit_month
        , count(*) as case_count
        , sum(a.initialfulladr_cases) as initial_adr_cnt
        , sum(a.persistentfulladr_cases) as persistent_adr_cnt
        , sum(a.p2p_full_ovtn) as p2p_ovrtn_cnt
        , sum(a.appeal_ovrtn_ind) as appeal_ovrtn_cnt
        , sum(a.mcr_ovtrn_ind) as mcr_ovrtn_cnt
        , sum(a.icm_md_reviewed_ind) as md_reviewed_cnt
    from tmp_1m.ec_avtar_25_26_3_od as a
    left join tmp_1y.tin_collection as d
        on substr(a.fa_prov_id, 2, 9) = d.tin
    where fin_brand in ('M&R', 'C&S')
        and (
            (ip_type in ('Medical', 'Surgical', 'Transplant')
             and to_varchar(admit_dt_act, 'MM/dd/yyyy') is not null)
            or ip_type in ('LTAC', 'SNF', 'AIR')
        )
        and fin_brand = 'M&R'
        and loc_flag = 1
        and (
            (global_cap = 'NA' and sgr_source_name = 'COSMOS'
             and fin_product_level_3 != 'INSTITUTIONAL' and tfm_include_flag = 1)
            or (sgr_source_name = 'NICE' and nce_tadm_dec_risk_type in ('FFS', 'PHYSICIAN'))
        )
        and hce_admit_month between '202301' and '202512'
    group by 1, 2, 3
""", engine)
engine.dispose()
print(f"Monthly (202301-202512): {df_monthly.shape}")

In [ ]:
# left_join member appeal onto core, fill missing counts with 0
df = df.merge(df_ma, on = ["hospital_group", "fin_market"], how = "left")

count_cols = ["case_count_2025", "initial_adr_cnt_2025", "persistent_adr_cnt", "p2p_ovrtn_cnt",
              "appeal_ovrtn_cnt", "mcr_ovrtn_cnt", "md_reviewed_cnt",
              "case_count_2026", "member_appeal_cnt", "initial_adr_cnt_2026"]
df[count_cols] = df[count_cols].fillna(0)

# eligibility: enough volume for the rates to mean something
df = df.query("case_count_2025 >= 30 and initial_adr_cnt_2025 >= 10 and initial_adr_cnt_2026 >= 10").copy()
print(f"After merge + eligibility filter: {df.shape}")

In [ ]:
# empirical Bayes shrinkage: adjusted = (numerator + k * global_rate) / (denominator + k)
# k = 50 pulls small-hospital rates toward the population rate so tiny samples don't
# produce extreme rates. Big hospitals barely move; small ones move a lot.

k = 50

# population-level rates (the "prior" each hospital gets pulled toward)
g_initial_adr_2025 = df["initial_adr_cnt_2025"].sum() / df["case_count_2025"].sum()
g_persistent_adr   = df["persistent_adr_cnt"].sum()   / df["case_count_2025"].sum()
g_p2p_overturn     = df["p2p_ovrtn_cnt"].sum()         / df["initial_adr_cnt_2025"].sum()
g_appeal_overturn  = df["appeal_ovrtn_cnt"].sum()       / df["initial_adr_cnt_2025"].sum()
g_mcr_overturn     = df["mcr_ovrtn_cnt"].sum()          / df["initial_adr_cnt_2025"].sum()
g_md_review        = df["md_reviewed_cnt"].sum()        / df["case_count_2025"].sum()
g_member_appeal    = df["member_appeal_cnt"].sum()      / df["initial_adr_cnt_2026"].sum()
g_initial_adr_2026 = df["initial_adr_cnt_2026"].sum()  / df["case_count_2026"].sum()

# like mutate() — one block, every adjusted rate
df = (
    df
    .assign(
        initial_adr_rate_2025 = lambda x: (x.initial_adr_cnt_2025 + k * g_initial_adr_2025) / (x.case_count_2025 + k),
        persistent_adr_rate   = lambda x: (x.persistent_adr_cnt   + k * g_persistent_adr)   / (x.case_count_2025 + k),
        p2p_overturn_rate     = lambda x: (x.p2p_ovrtn_cnt         + k * g_p2p_overturn)     / (x.initial_adr_cnt_2025 + k),
        appeal_overturn_rate  = lambda x: (x.appeal_ovrtn_cnt       + k * g_appeal_overturn)  / (x.initial_adr_cnt_2025 + k),
        mcr_overturn_rate     = lambda x: (x.mcr_ovrtn_cnt          + k * g_mcr_overturn)     / (x.initial_adr_cnt_2025 + k),
        md_review_rate        = lambda x: (x.md_reviewed_cnt        + k * g_md_review)        / (x.case_count_2025 + k),
        member_appeal_rate    = lambda x: (x.member_appeal_cnt      + k * g_member_appeal)    / (x.initial_adr_cnt_2026 + k),
        initial_adr_rate_2026 = lambda x: (x.initial_adr_cnt_2026  + k * g_initial_adr_2026) / (x.case_count_2026 + k),
        # log-volume features — lets IF tell "weird behavior" apart from "tiny sample"
        log_case_count_2025      = lambda x: np.log1p(x.case_count_2025),
        log_initial_adr_cnt_2025 = lambda x: np.log1p(x.initial_adr_cnt_2025),
        log_initial_adr_cnt_2026 = lambda x: np.log1p(x.initial_adr_cnt_2026)
    )
)

# the feature lists used everywhere below
rate_kpi = ["initial_adr_rate_2025", "persistent_adr_rate", "p2p_overturn_rate",
            "appeal_overturn_rate", "mcr_overturn_rate", "md_review_rate",
            "member_appeal_rate", "initial_adr_rate_2026"]
iso_kpi = ["log_case_count_2025", "log_initial_adr_cnt_2025", "log_initial_adr_cnt_2026"] + rate_kpi

print(f"Shape: {df.shape}")

In [ ]:
df[rate_kpi].describe().round(4)

In [ ]:
df[rate_kpi].hist(bins = 20, figsize = (16, 8), edgecolor = "white")
plt.suptitle("Hospital Group — Bayes-Adjusted Rate Distributions")
plt.tight_layout()
plt.show()

## Section A — National Isolation Forest

Same model as the original notebook: IF on the Bayes-adjusted rates + log volumes.
Flags the ~5% most unusual hospital groups nationally.

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(df[iso_kpi])

iso = IsolationForest(contamination = 0.05, n_estimators = 300, random_state = 42)
iso.fit(X)

df = (
    df
    .assign(
        anomaly_score = iso.decision_function(X),
        anomaly_flag  = iso.predict(X) == -1
    )
)
df["anomaly_rank"] = df["anomaly_score"].rank(method = "dense").astype(int)

print(f"Flagged nationally: {df['anomaly_flag'].sum()} of {len(df)}")

In [ ]:
# MAD z-scores — robust version of z-scores (median/MAD instead of mean/SD)
# used only to explain WHY a hospital was flagged, not as model input
for col in iso_kpi:
    med = df[col].median()
    mad = median_abs_deviation(df[col], scale = 1.0) * 1.4826  # 1.4826 makes MAD ~ SD
    df[col + "_z"] = (df[col] - med) / mad if mad > 0 else 0

z_cols = [col + "_z" for col in iso_kpi]

# the driver = the feature with the biggest |z| for each hospital
df = (
    df
    .assign(
        driver    = lambda x: x[z_cols].abs().idxmax(axis = 1).str.removesuffix("_z"),
        z_score   = lambda x: x[z_cols].apply(lambda r: r[r.abs().idxmax()], axis = 1),
        direction = lambda x: np.where(x.z_score > 0, "high", "low"),
        reason    = lambda x: np.where(x.anomaly_flag, x.driver + " (" + x.direction + ")", "")
    )
)

# when the driver is a log feature, report the raw count instead (easier to read)
df["driver_raw"] = df["driver"].replace({
    "log_case_count_2025":      "case_count_2025",
    "log_initial_adr_cnt_2025": "initial_adr_cnt_2025",
    "log_initial_adr_cnt_2026": "initial_adr_cnt_2026"
})

# look up each hospital's own value for its driver column, and the population median
medians = df[rate_kpi + ["case_count_2025", "initial_adr_cnt_2025", "initial_adr_cnt_2026"]].median()
df["driver_value"]      = df.apply(lambda r: r[r["driver_raw"]], axis = 1)
df["driver_pop_median"] = df["driver_raw"].map(medians)
df["driver_dev"]        = (df["driver_value"] - df["driver_pop_median"]) / df["driver_pop_median"]

print("Drivers computed.")

In [ ]:
# flagged hospitals, most anomalous first
(
    df
    .query("anomaly_flag == True")
    .sort_values("anomaly_score")
    [["hospital_group", "fin_market", "case_count_2025", "initial_adr_cnt_2025",
      *rate_kpi, "anomaly_score", "anomaly_rank", "reason",
      "driver_value", "driver_pop_median", "driver_dev"]]
    .round(4)
)

## Section B — LOF + Ensemble Anomaly Score

LOF (Local Outlier Factor) is a second, independent anomaly detector — it's density-based
(how isolated is this point from its neighbors?) where IF is split-based. A hospital that
scores high on BOTH is a higher-confidence flag. We min-max both scores to 0-1 and average.

In [ ]:
lof = LocalOutlierFactor(n_neighbors = 20, contamination = 0.05)
df["lof_flag"]  = lof.fit_predict(X) == -1
df["lof_score"] = lof.negative_outlier_factor_  # more negative = more anomalous

# rescale both scores to 0-1 where 1 = most anomalous (flip sign first so high = bad)
if_flipped  = -df["anomaly_score"]
lof_flipped = -df["lof_score"]
df["norm_if_score"]  = (if_flipped  - if_flipped.min())  / (if_flipped.max()  - if_flipped.min())
df["norm_lof_score"] = (lof_flipped - lof_flipped.min()) / (lof_flipped.max() - lof_flipped.min())

df["ensemble_score"] = (df["norm_if_score"] + df["norm_lof_score"]) / 2
df["ensemble_flag"]  = df["ensemble_score"] >= df["ensemble_score"].quantile(0.95)

print(f"Flagged by IF and ensemble: {(df['anomaly_flag'] & df['ensemble_flag']).sum()}")
print(f"Flagged by IF only:         {(df['anomaly_flag'] & ~df['ensemble_flag']).sum()}")
print(f"Flagged by LOF only:        {(df['lof_flag'] & ~df['anomaly_flag']).sum()}")

(
    df
    .nlargest(10, "ensemble_score")
    [["hospital_group", "fin_market", "anomaly_score", "lof_score",
      "norm_if_score", "norm_lof_score", "ensemble_score",
      "anomaly_flag", "lof_flag", "ensemble_flag", "reason"]]
    .round(4)
)

## Section C — K-Means Clustering + PCA

K-means groups hospitals into behavioral archetypes (e.g. "high ADR + low overturn").
PCA squashes the 11 features into 2 dimensions so we can see the whole population
on one scatter plot, with clusters colored and IF anomalies marked.

In [ ]:
# elbow plot — look for where the curve flattens to pick k
inertias = []
for n in range(2, 9):
    km = KMeans(n_clusters = n, random_state = 42, n_init = 10)
    km.fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize = (8, 4))
plt.plot(range(2, 9), inertias, "o-", color = "steelblue")
plt.xlabel("k (number of clusters)")
plt.ylabel("Inertia (within-cluster spread)")
plt.title("K-Means Elbow — Hospital Groups")
plt.tight_layout()
plt.show()

In [ ]:
km = KMeans(n_clusters = 5, random_state = 42, n_init = 10)
df["cluster_label"] = km.fit_predict(X)

print(df["cluster_label"].value_counts().sort_index())

# median rate profile per cluster — like group_by(cluster) %>% summarise(across(everything(), median))
df.groupby("cluster_label")[rate_kpi].median().round(4)

In [ ]:
pca = PCA(n_components = 2)
pcs = pca.fit_transform(X)
df["pc1"] = pcs[:, 0]
df["pc2"] = pcs[:, 1]
print(f"Variance explained: PC1 = {pca.explained_variance_ratio_[0]:.1%}, PC2 = {pca.explained_variance_ratio_[1]:.1%}")

fig, ax = plt.subplots(figsize = (12, 8))
colors = plt.cm.tab10(range(5))

for n in range(5):
    sub = df[df["cluster_label"] == n]
    ax.scatter(sub["pc1"], sub["pc2"], color = colors[n], alpha = 0.45, s = 18, label = f"Cluster {n}")

# IF anomalies on top as red X
sub = df[df["anomaly_flag"]]
ax.scatter(sub["pc1"], sub["pc2"], color = "red", marker = "x", s = 60, linewidths = 1.5,
           label = "IF Anomaly", zorder = 5)

# name the top 5 most anomalous
for _, row in df.nsmallest(5, "anomaly_score").iterrows():
    ax.annotate(str(row["hospital_group"])[:20], (row["pc1"], row["pc2"]),
                fontsize = 7, color = "red", xytext = (5, 5), textcoords = "offset points")

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.set_title("PCA — Hospital Groups by Cluster (IF Anomalies in Red)")
ax.legend(loc = "upper right", fontsize = 9)
plt.tight_layout()
plt.show()

## Section D — Within-Market IF

Re-run IF inside each market, so hospitals are compared to their regional peers
instead of the whole country. Bayes shrinkage is recomputed with the MARKET rate
as the prior. Contamination is 10% here because markets are small (~20-60 hospitals).

In [ ]:
# only markets with enough hospitals for a within-market model
market_sizes = df["fin_market"].value_counts()
big_markets = market_sizes[market_sizes >= 20].index.tolist()
print(f"Markets with >= 20 hospitals: {len(big_markets)}")

# like group_by(fin_market) %>% group_map(~ fit IF on each group)
local_pieces = []
for mkt in big_markets:
    sub = df[df["fin_market"] == mkt].copy()

    # market-level rates = the new prior for shrinkage
    m_initial_adr_2025 = sub["initial_adr_cnt_2025"].sum() / sub["case_count_2025"].sum()
    m_persistent_adr   = sub["persistent_adr_cnt"].sum()   / sub["case_count_2025"].sum()
    m_p2p_overturn     = sub["p2p_ovrtn_cnt"].sum()         / sub["initial_adr_cnt_2025"].sum()
    m_appeal_overturn  = sub["appeal_ovrtn_cnt"].sum()       / sub["initial_adr_cnt_2025"].sum()
    m_mcr_overturn     = sub["mcr_ovrtn_cnt"].sum()          / sub["initial_adr_cnt_2025"].sum()
    m_md_review        = sub["md_reviewed_cnt"].sum()        / sub["case_count_2025"].sum()
    m_member_appeal    = sub["member_appeal_cnt"].sum()      / sub["initial_adr_cnt_2026"].sum()
    m_initial_adr_2026 = sub["initial_adr_cnt_2026"].sum()  / sub["case_count_2026"].sum()

    sub = (
        sub
        .assign(
            initial_adr_rate_2025 = lambda x: (x.initial_adr_cnt_2025 + k * m_initial_adr_2025) / (x.case_count_2025 + k),
            persistent_adr_rate   = lambda x: (x.persistent_adr_cnt   + k * m_persistent_adr)   / (x.case_count_2025 + k),
            p2p_overturn_rate     = lambda x: (x.p2p_ovrtn_cnt         + k * m_p2p_overturn)     / (x.initial_adr_cnt_2025 + k),
            appeal_overturn_rate  = lambda x: (x.appeal_ovrtn_cnt       + k * m_appeal_overturn)  / (x.initial_adr_cnt_2025 + k),
            mcr_overturn_rate     = lambda x: (x.mcr_ovrtn_cnt          + k * m_mcr_overturn)     / (x.initial_adr_cnt_2025 + k),
            md_review_rate        = lambda x: (x.md_reviewed_cnt        + k * m_md_review)        / (x.case_count_2025 + k),
            member_appeal_rate    = lambda x: (x.member_appeal_cnt      + k * m_member_appeal)    / (x.initial_adr_cnt_2026 + k),
            initial_adr_rate_2026 = lambda x: (x.initial_adr_cnt_2026  + k * m_initial_adr_2026) / (x.case_count_2026 + k)
        )
    )

    X_sub = StandardScaler().fit_transform(sub[iso_kpi])
    iso_sub = IsolationForest(contamination = 0.10, n_estimators = 200, random_state = 42)
    iso_sub.fit(X_sub)

    sub["local_anomaly_score"] = iso_sub.decision_function(X_sub)
    sub["local_anomaly_flag"]  = iso_sub.predict(X_sub) == -1
    local_pieces.append(sub[["hospital_group", "fin_market", "local_anomaly_score", "local_anomaly_flag"]])

# stack the per-market results back together — like bind_rows()
df_local = pd.concat(local_pieces, ignore_index = True)
df = df.merge(df_local, on = ["hospital_group", "fin_market"], how = "left")
df["local_anomaly_flag"] = df["local_anomaly_flag"].fillna(False)

print(f"Locally flagged: {df['local_anomaly_flag'].sum()}")

In [ ]:
# 2x2: who's an outlier nationally vs. within their own market?
# "both" = strongest signal; "local only" = hidden by national averages
pd.crosstab(
    df["anomaly_flag"].map({True: "national outlier", False: "national normal"}),
    df["local_anomaly_flag"].map({True: "local outlier", False: "local normal"}),
    margins = True
)

In [ ]:
# the interesting quadrant: local outliers that the national model missed
(
    df
    .query("local_anomaly_flag == True and anomaly_flag == False")
    .sort_values("local_anomaly_score")
    [["hospital_group", "fin_market", "case_count_2025", *rate_kpi, "local_anomaly_score"]]
    .round(4)
    .head(20)
)

## Section E — Market-Level IF + KS Test

Roll everything up to the market level: which MARKETS look anomalous as a whole?
Then a KS test per market x feature checks whether each market's distribution of
hospital rates is statistically different from the national distribution.

In [ ]:
# market profile = median of its hospitals' adjusted rates
df_market = (
    df
    .groupby("fin_market")[iso_kpi]
    .median()
    .reset_index()
)

X_market = StandardScaler().fit_transform(df_market[iso_kpi])
iso_market = IsolationForest(contamination = 0.10, n_estimators = 200, random_state = 42)
iso_market.fit(X_market)

df_market = (
    df_market
    .assign(
        market_anomaly_score = iso_market.decision_function(X_market),
        market_anomaly_flag  = iso_market.predict(X_market) == -1
    )
)

print(f"Markets flagged: {df_market['market_anomaly_flag'].sum()} of {len(df_market)}")
(
    df_market
    .sort_values("market_anomaly_score")
    [["fin_market", *rate_kpi, "market_anomaly_score", "market_anomaly_flag"]]
    .round(4)
    .head(15)
)

In [ ]:
# KS test: is this market's distribution of hospital rates different from the nation's?
# (two nested loops — like crossing(market, feature) %>% map(~ ks.test(...)))
ks_rows = []
for mkt in big_markets:
    sub = df[df["fin_market"] == mkt]
    for col in rate_kpi:
        stat, p = ks_2samp(sub[col], df[col])
        ks_rows.append({
            "fin_market": mkt,
            "feature": col,
            "ks_stat": stat,
            "p_value": p,
            "market_median": sub[col].median(),
            "national_median": df[col].median()
        })

df_ks = (
    pd.DataFrame(ks_rows)
    .assign(
        direction   = lambda x: np.where(x.market_median > x.national_median, "high", "low"),
        significant = lambda x: x.p_value < 0.05
    )
)

print(f"Significant market-feature combos (p < 0.05): {df_ks['significant'].sum()} of {len(df_ks)}")
(
    df_ks
    .query("significant == True")
    .sort_values("ks_stat", ascending = False)
    .round(4)
    .head(30)
)

## Section F — Monthly Trend Detection

The IF models are snapshots. Here we fit a straight line through each hospital's
monthly rates (202301+) and flag hospitals whose slope is in the worst 5% — an early
warning list of hospitals heading toward anomalous territory.

Note: member_appeal is excluded (only exists from 202601, too short for a trend).

In [ ]:
# keep only hospitals in our main eligible table — like semi_join()
df_monthly = df_monthly.merge(df[["hospital_group", "fin_market"]], on = ["hospital_group", "fin_market"], how = "inner")

# month_index = 1, 2, 3, ... so the regression x-axis is evenly spaced
df_monthly["month_index"] = df_monthly["hce_admit_month"].rank(method = "dense").astype(int)

# Bayes shrinkage at monthly grain — k = 20 (smaller, monthly volumes are smaller)
k_month = 20
gm_initial_adr    = df_monthly["initial_adr_cnt"].sum()   / df_monthly["case_count"].sum()
gm_persistent_adr = df_monthly["persistent_adr_cnt"].sum() / df_monthly["case_count"].sum()
gm_p2p_overturn   = df_monthly["p2p_ovrtn_cnt"].sum()      / df_monthly["initial_adr_cnt"].sum()
gm_appeal_overturn= df_monthly["appeal_ovrtn_cnt"].sum()    / df_monthly["initial_adr_cnt"].sum()
gm_mcr_overturn   = df_monthly["mcr_ovtrn_cnt"].sum()       / df_monthly["initial_adr_cnt"].sum()
gm_md_review      = df_monthly["md_reviewed_cnt"].sum()     / df_monthly["case_count"].sum()

df_monthly = (
    df_monthly
    .assign(
        initial_adr_rate     = lambda x: (x.initial_adr_cnt   + k_month * gm_initial_adr)     / (x.case_count + k_month),
        persistent_adr_rate  = lambda x: (x.persistent_adr_cnt + k_month * gm_persistent_adr) / (x.case_count + k_month),
        p2p_overturn_rate    = lambda x: (x.p2p_ovrtn_cnt      + k_month * gm_p2p_overturn)   / (x.initial_adr_cnt + k_month),
        appeal_overturn_rate = lambda x: (x.appeal_ovrtn_cnt    + k_month * gm_appeal_overturn)/ (x.initial_adr_cnt + k_month),
        mcr_overturn_rate    = lambda x: (x.mcr_ovtrn_cnt       + k_month * gm_mcr_overturn)   / (x.initial_adr_cnt + k_month),
        md_review_rate       = lambda x: (x.md_reviewed_cnt     + k_month * gm_md_review)      / (x.case_count + k_month)
    )
)

print(f"Monthly rows after filter: {df_monthly.shape}")
df_monthly.head()

In [ ]:
# fit a line per hospital per KPI — slope = direction of travel
# like group_by(hospital) %>% group_map(~ lm(rate ~ month, .x))
monthly_kpi = ["initial_adr_rate", "persistent_adr_rate", "p2p_overturn_rate",
               "appeal_overturn_rate", "mcr_overturn_rate", "md_review_rate"]

slope_rows = []
for (hosp, mkt), grp in df_monthly.groupby(["hospital_group", "fin_market"]):
    if len(grp) < 6:  # need at least 6 months for a meaningful slope
        continue
    grp = grp.sort_values("month_index")
    row = {"hospital_group": hosp, "fin_market": mkt, "n_months": len(grp)}
    for col in monthly_kpi:
        result = linregress(grp["month_index"], grp[col])
        row[col + "_slope"] = result.slope
    slope_rows.append(row)

df_slopes = pd.DataFrame(slope_rows)
print(f"Hospitals with slopes: {len(df_slopes)}")

# worsening = ADR/MD-review rising, or overturn rates falling (overturns dropping
# means denials are sticking more)
df_slopes = (
    df_slopes
    .assign(
        initial_adr_worsening     = lambda x: x.initial_adr_rate_slope     >= x.initial_adr_rate_slope.quantile(0.95),
        persistent_adr_worsening  = lambda x: x.persistent_adr_rate_slope  >= x.persistent_adr_rate_slope.quantile(0.95),
        md_review_worsening       = lambda x: x.md_review_rate_slope       >= x.md_review_rate_slope.quantile(0.95),
        p2p_overturn_worsening    = lambda x: x.p2p_overturn_rate_slope    <= x.p2p_overturn_rate_slope.quantile(0.05),
        appeal_overturn_worsening = lambda x: x.appeal_overturn_rate_slope <= x.appeal_overturn_rate_slope.quantile(0.05),
        mcr_overturn_worsening    = lambda x: x.mcr_overturn_rate_slope    <= x.mcr_overturn_rate_slope.quantile(0.05)
    )
)

worsening_cols = ["initial_adr_worsening", "persistent_adr_worsening", "md_review_worsening",
                  "p2p_overturn_worsening", "appeal_overturn_worsening", "mcr_overturn_worsening"]
df_slopes["trend_flag"]       = df_slopes[worsening_cols].any(axis = 1)
df_slopes["trend_flag_count"] = df_slopes[worsening_cols].sum(axis = 1)

print(f"Hospitals with any worsening trend: {df_slopes['trend_flag'].sum()}")
(
    df_slopes
    .query("trend_flag == True")
    .sort_values("trend_flag_count", ascending = False)
    .round(5)
    .head(20)
)

In [ ]:
# the 5 hospitals with the fastest-rising ADR rate, plotted over time
top5_trend = df_slopes.nlargest(5, "initial_adr_rate_slope")

fig, ax = plt.subplots(figsize = (12, 5))
colors = plt.cm.Set1(range(5))

for i, (_, row) in enumerate(top5_trend.iterrows()):
    grp = (
        df_monthly
        .query("hospital_group == @row.hospital_group and fin_market == @row.fin_market")
        .sort_values("hce_admit_month")
    )
    ax.plot(grp["hce_admit_month"].astype(str), grp["initial_adr_rate"],
            marker = "o", linewidth = 2, color = colors[i],
            label = f"{str(row['hospital_group'])[:25]} ({row['fin_market']})")

ax.set_xlabel("Month")
ax.set_ylabel("Initial ADR Rate (Bayes-adjusted)")
ax.set_title("Top 5 Hospitals — Fastest-Rising Initial ADR Rate")
ax.legend(fontsize = 9)
plt.xticks(rotation = 45, ha = "right")
plt.tight_layout()
plt.show()

## Section G — Nearest-Neighbor Peer Group

For each flagged hospital, find the 5 most similar NON-flagged hospitals (same feature
space the IF used). "This hospital looks like these 5 normal ones, except for X" —
useful for investigating and packaging the anomalies.

In [ ]:
df_flagged = df[df["anomaly_flag"]].reset_index(drop = True)
df_normal  = df[~df["anomaly_flag"]].reset_index(drop = True)

# scale on the full population so flagged and normal are on the same scale
X_flagged = scaler.transform(df_flagged[iso_kpi])
X_normal  = scaler.transform(df_normal[iso_kpi])

nn = NearestNeighbors(n_neighbors = 5)
nn.fit(X_normal)
distances, neighbor_idx = nn.kneighbors(X_flagged)

# one row per flagged hospital: its rates vs. the mean of its 5 peers
peer_rows = []
for i in range(len(df_flagged)):
    peers = df_normal.iloc[neighbor_idx[i]]
    row = {
        "hospital_group":    df_flagged.loc[i, "hospital_group"],
        "fin_market":        df_flagged.loc[i, "fin_market"],
        "anomaly_rank":      df_flagged.loc[i, "anomaly_rank"],
        "reason":            df_flagged.loc[i, "reason"],
        "avg_peer_distance": distances[i].mean(),
        "peer_1": peers.iloc[0]["hospital_group"],
        "peer_2": peers.iloc[1]["hospital_group"],
        "peer_3": peers.iloc[2]["hospital_group"]
    }
    for col in rate_kpi:
        row["entity_" + col] = df_flagged.loc[i, col]
        row["peers_" + col]  = peers[col].mean()
        row["diff_" + col]   = df_flagged.loc[i, col] - peers[col].mean()
    peer_rows.append(row)

df_peers = pd.DataFrame(peer_rows).sort_values("anomaly_rank")
diff_cols = ["diff_" + col for col in rate_kpi]

print(f"Peer comparisons built: {len(df_peers)}")
df_peers[["hospital_group", "fin_market", "anomaly_rank", "reason",
          "avg_peer_distance", *diff_cols]].round(4).head(15)

## Section H — Export

In [ ]:
# bring trend flags onto the main table
df = df.merge(
    df_slopes[["hospital_group", "fin_market", "trend_flag", "trend_flag_count"]],
    on = ["hospital_group", "fin_market"],
    how = "left"
)
df["trend_flag"]       = df["trend_flag"].fillna(False)
df["trend_flag_count"] = df["trend_flag_count"].fillna(0)

output_dir = r"C:\Users\knguy139\Documents\Projects\Data\Output"

df.to_csv(output_dir + r"\loc_if_hospital_group_expanded.csv", index = False)

(
    df
    .query("anomaly_flag == True")
    .sort_values("anomaly_score")
    [["hospital_group", "fin_market", "case_count_2025", "initial_adr_cnt_2025",
      *rate_kpi, "anomaly_score", "anomaly_rank", "reason",
      "ensemble_score", "ensemble_flag", "cluster_label",
      "local_anomaly_flag", "trend_flag", "trend_flag_count"]]
    .round(4)
    .to_csv(output_dir + r"\loc_if_hospital_group_flagged_expanded.csv", index = False)
)

df_market.to_csv(output_dir + r"\loc_if_market_summary.csv", index = False)
df_ks.to_csv(output_dir + r"\loc_if_market_ks_test.csv", index = False)
df_slopes.to_csv(output_dir + r"\loc_if_trend_flags.csv", index = False)
df_peers.to_csv(output_dir + r"\loc_if_peer_comparison.csv", index = False)

print(f"Exported 6 files to {output_dir}")

## Section I — Changepoint Detection

The trend section fits straight lines, but hospitals usually don't drift — they SHIFT
abruptly (new UM vendor, staffing change, policy change). Here we find the single
biggest level shift in each hospital's monthly ADR rate and when it happened.
The "when" is the actionable part: you can ask what changed at that hospital that month.

One-time install: `pip install ruptures`

In [ ]:
import ruptures as rpt

# find the single biggest level shift per hospital — like group_by() %>% group_map()
cp_rows = []
for (hosp, mkt), grp in df_monthly.groupby(["hospital_group", "fin_market"]):
    if len(grp) < 12:  # need enough months on both sides of a potential shift
        continue
    grp = grp.sort_values("month_index")
    y = grp["initial_adr_rate"].values

    # Binseg with n_bkps = 1: "where should I split this series into two levels
    # so the split explains the most variance?"
    algo = rpt.Binseg(model = "l2").fit(y)
    bkp = algo.predict(n_bkps = 1)[0]  # index of the first month AFTER the shift

    # skip shifts at the edges — fewer than 3 months on one side isn't a level, it's noise
    if bkp < 3 or bkp > len(y) - 3:
        continue

    cp_rows.append({
        "hospital_group": hosp,
        "fin_market":     mkt,
        "n_months":       len(grp),
        "cp_month":       grp["hce_admit_month"].iloc[bkp],  # first month of the new level
        "mean_before":    y[:bkp].mean(),
        "mean_after":     y[bkp:].mean()
    })

df_cp = (
    pd.DataFrame(cp_rows)
    .assign(
        level_shift = lambda x: x.mean_after - x.mean_before,
        abs_shift   = lambda x: x.level_shift.abs(),
        pct_shift   = lambda x: x.level_shift / x.mean_before
    )
    .assign(cp_flag = lambda x: x.abs_shift >= x.abs_shift.quantile(0.95))
    .sort_values("abs_shift", ascending = False)
)

print(f"Hospitals with a detectable changepoint: {len(df_cp)}")
print(f"Flagged (top 5% biggest shifts): {df_cp['cp_flag'].sum()}")
df_cp.round(4).head(20)

In [ ]:
# the 5 biggest level shifts, plotted — dashed line marks the changepoint month
top5_cp = df_cp.head(5)

fig, axes = plt.subplots(5, 1, figsize = (12, 16))

for ax, (_, row) in zip(axes, top5_cp.iterrows()):
    grp = (
        df_monthly
        .query("hospital_group == @row.hospital_group and fin_market == @row.fin_market")
        .sort_values("hce_admit_month")
    )
    months = grp["hce_admit_month"].astype(str)
    ax.plot(months, grp["initial_adr_rate"], marker = "o", color = "steelblue")
    ax.axvline(str(row["cp_month"]), color = "red", linestyle = "--", linewidth = 1.5)

    # before/after level lines
    before = months < str(row["cp_month"])
    ax.hlines(row["mean_before"], xmin = 0, xmax = before.sum() - 1, color = "gray", linestyle = ":")
    ax.hlines(row["mean_after"], xmin = before.sum(), xmax = len(months) - 1, color = "gray", linestyle = ":")

    ax.set_title(
        f"{str(row['hospital_group'])[:40]} ({row['fin_market']}) — "
        f"shift {row['level_shift']:+.3f} at {row['cp_month']}",
        fontsize = 11, fontweight = "bold"
    )
    ax.set_ylabel("Initial ADR Rate")
    ax.tick_params(axis = "x", rotation = 45, labelsize = 7)

plt.tight_layout()
plt.show()

## Section J — SHAP Driver Attribution

The MAD z-score driver answers "which single feature is most extreme for this hospital?"
SHAP answers a better question: "which features actually pushed THIS hospital's IF score
toward anomalous?" — including hospitals flagged for unusual COMBINATIONS where no
single z-score stands out.

Sign convention: lower IF score = more anomalous, so NEGATIVE SHAP values push toward anomalous.

One-time install: `pip install shap`

In [ ]:
import shap

explainer = shap.TreeExplainer(iso)
shap_vals = explainer.shap_values(X)  # one row per hospital, one column per feature

df_shap = pd.DataFrame(shap_vals, columns = iso_kpi, index = df.index)

# the SHAP driver = the feature pushing hardest toward anomalous (most negative)
df["shap_driver"]       = df_shap.idxmin(axis = 1)
df["shap_driver_value"] = df_shap.min(axis = 1)

# how often does SHAP agree with the MAD z-score driver on flagged hospitals?
flagged = df[df["anomaly_flag"]]
agreement = (flagged["shap_driver"] == flagged["driver"]).mean()
print(f"SHAP driver == MAD z driver on flagged hospitals: {agreement:.0%}")

# where they disagree, SHAP is seeing a combination effect the z-score missed
(
    flagged
    .query("shap_driver != driver")
    [["hospital_group", "fin_market", "anomaly_rank", "driver", "shap_driver", "z_score", "shap_driver_value"]]
    .sort_values("anomaly_rank")
    .round(4)
    .head(15)
)

In [ ]:
# beeswarm: every hospital's SHAP value per feature.
# points far LEFT = pushing toward anomalous; color = feature value (scaled)
shap.summary_plot(shap_vals, features = X, feature_names = iso_kpi, show = False)
plt.title("SHAP — What Drives Anomaly Scores Across All Hospitals")
plt.tight_layout()
plt.show()

In [ ]:
# per-hospital breakdown for the top 5 anomalies — like the deviation bar chart,
# but showing the model's actual reasoning instead of raw deviations
top5_anom = df.nsmallest(5, "anomaly_score")

fig, axes = plt.subplots(5, 1, figsize = (10, 16))

for ax, (idx, row) in zip(axes, top5_anom.iterrows()):
    vals = df_shap.loc[idx].sort_values()
    colors = ["#d62728" if v < 0 else "#2ca02c" for v in vals]
    ax.barh(vals.index, vals.values, color = colors, alpha = 0.85, edgecolor = "white")
    ax.axvline(0, color = "black", linewidth = 0.8)
    ax.set_title(
        f"{str(row['hospital_group'])[:40]} ({row['fin_market']}) — rank #{row['anomaly_rank']}",
        fontsize = 11, fontweight = "bold"
    )
    ax.set_xlabel("SHAP value (negative = pushes toward anomalous)", fontsize = 9)
    ax.tick_params(axis = "y", labelsize = 8)

plt.tight_layout()
plt.show()

In [ ]:
# re-export the full table (now includes shap_driver columns) + changepoints
df.to_csv(output_dir + r"\loc_if_hospital_group_expanded.csv", index = False)
df_cp.to_csv(output_dir + r"\loc_if_changepoints.csv", index = False)
print("Re-exported full table + changepoints.")